### 1. Config and Environment

In [9]:
!pip install chronos-forecasting matplotlib > /dev/null 2>&1
import os
import pandas as pd
from google.colab import drive
from chronos import BaseChronosPipeline
import torch
from torch import nn
from torch.utils.data import DataLoader
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import numpy as np

In [3]:
CONFIG = {
    "seed": 42,
    "max_rul": 125,
    "window_size": 30,
    "batch_size": 256,
    "data_fractions": [0.1, 0.25, 0.5, 0.75, 1.0],
    "chronos_model": "amazon/chronos-2"
}

### 2. Data Ingestion and Preprocessing

In [8]:
# Data Loading
if not os.path.exists('/content/drive'):
  drive.mount('/content/drive')

test_file_path = '/content/drive/MyDrive/Predictive Maintenance LSTM/CMAPSSData/test_FD001.txt'
test_rul_file_path = '/content/drive/MyDrive/Predictive Maintenance LSTM/CMAPSSData/RUL_FD001.txt'
train_file_path = '/content/drive/MyDrive/Predictive Maintenance LSTM/CMAPSSData/train_FD001.txt'

# CMAPSS Column Names
# 0: unit, 1: time, 2-4: settings, 5-25: sensors
col_names = ['unit_number', 'time_cycles', 'setting_1', 'setting_2', 'setting_3'] + [f's_{i}' for i in range(1, 22)]

df_test = pd.read_csv(test_file_path, sep=r'\s+', header=None, names=col_names)
df_train = pd.read_csv(train_file_path, sep=r'\s+', header=None, names=col_names)

# RUL Calculation (Train)
train_max_cycles = df_train.groupby('unit_number')['time_cycles'].max().reset_index()
train_max_cycles.columns = ['unit_number', 'max_cycles']
df_train = df_train.merge(train_max_cycles, on='unit_number', how='left')

# Actual RUL = max_cycles - current_cycle
df_train['actual_rul'] = df_train['max_cycles'] - df_train['time_cycles']

df_train['clipped_rul'] = df_train['actual_rul'].clip(upper=CONFIG["max_rul"])

# RUL Calculation (Test)
test_rul_ground_truth = pd.read_csv(test_rul_file_path, header=None, names=['remaining_cycles'])
test_rul_ground_truth['unit_number'] = range(1, len(test_rul_ground_truth) + 1)

test_max_cycles = df_test.groupby('unit_number')['time_cycles'].max().reset_index()
test_max_cycles.columns = ['unit_number', 'max_cycles']

df_test = df_test.merge(test_max_cycles, on='unit_number', how='left')
df_test = df_test.merge(test_rul_ground_truth, on='unit_number', how='left')

# Calculate actual RUL for each time step
df_test['actual_rul'] = df_test['remaining_cycles'] + (df_test['max_cycles'] - df_test['time_cycles'])

# Clip RUL
df_test['clipped_rul'] = df_test['actual_rul'].clip(upper=CONFIG["max_rul"])

# Trim useful columns
# 0: unit, 1: time
# 9: T50 (s_4), 16: Ps30 (s_11), 12: P30 (s_7), 14: Nc (s_9), 17: phi (s_12)
# Note: indices in original 0-25 schema: [0, 1, 9, 16, 12, 14, 17]
useful_cols_indices = [0, 1, 9, 16, 12, 14, 17]
useful_col_names = [col_names[i] for i in useful_cols_indices]

train = df_train[useful_col_names].values
test = df_test[useful_col_names].values

print(f"Selected columns: {useful_col_names}")
print(f"Trimmed train shape: {train.shape}")
print(f"Trimmed test shape: {test.shape}")
display(df_train[['unit_number', 'time_cycles', 'actual_rul', 'clipped_rul']].head())

# Windowing
def generate_windows(df, sensor_cols, window_size=CONFIG["window_size"],  target_col="clipped_rul"):
    windows = []
    labels = []

    # Iterate through each engine
    for unit_id, unit_data in df.groupby("unit_number"):
        sensor_vals = unit_data[sensor_cols].values
        target_vals = unit_data[target_col].values

        # Create windows
        for i in range(len(unit_data) - window_size + 1):
            window = sensor_vals[i : i + window_size]
            label = target_vals[i + window_size - 1]

            windows.append(window)
            labels.append(label)

    # Return array of windows and labels
    return np.array(windows), np.array(labels)

train_windows, train_labels = generate_windows(df_train, sensor_cols=useful_col_names[2:])
test_windows, test_labels = generate_windows(df_test, sensor_cols=useful_col_names[2:])

print("-------------------")
print(f"Train windows shape: {train_windows.shape}")
print(f"Train labels shape: {train_labels.shape}")
display(df_train[['unit_number', 'time_cycles', 'actual_rul', 'clipped_rul']].head())

Selected columns: ['unit_number', 'time_cycles', 's_5', 's_12', 's_8', 's_10', 's_13']
Trimmed train shape: (20631, 7)
Trimmed test shape: (13096, 7)


,unit_number,time_cycles,actual_rul,clipped_rul
0,1,1,191,125
1,1,2,190,125
2,1,3,189,125
3,1,4,188,125
4,1,5,187,125


-------------------
Train windows shape: (17731, 30, 5)
Train labels shape: (17731,)


,unit_number,time_cycles,actual_rul,clipped_rul
0,1,1,191,125
1,1,2,190,125
2,1,3,189,125
3,1,4,188,125
4,1,5,187,125


### Model Setup

In [10]:
pipeline = BaseChronosPipeline.from_pretrained("amazon/chronos-2", device_map="cuda")
model = pipeline.model



Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]